# [01-1] 데이터 탐색과 시각화 — EOL 최종검사 불량 선별

**K-뉴딜 아카데미 스마트팩토리 캡스톤 · 공통 미니 프로젝트 (1~2회차 · 4개 팀 동일 데이터)**

| 데이터 | 분석단위 | 규모 | 기간 |
|---|---|---|---|
| `K-뉴딜_아카데미_1회차_공통MiniCase_미니프로젝트_데이터셋_수정.xlsx` | 제품 1대의 조립 + 최종검사 기록 1건 | 1,200행 × 20열 | 2026년 7월 1일 ~ 7월 31일 |

### 오늘의 목표

1. 데이터의 **크기·구조·결측**을 확인한다
2. **어느 조건에서 불량이 많은지** 그룹별로 비교한다
3. **두 조건을 겹쳐 보고**(교차표) 성급한 결론을 피한다
4. 그래프로 **공정규격과 실제 분포**를 확인한다

> 분석 시간의 절반 이상은 모델이 아니라 **데이터를 이해하는 데** 씁니다.
> 코드를 외우려 하지 말고, **출력된 숫자가 현장에서 무슨 뜻인지**에 집중하세요.

### 실행 방법
셀을 클릭하고 `Shift + Enter`. 위에서부터 순서대로 실행합니다.

---

## 0. 현장 상황

소형 전자제품 조립공장입니다. 제품은 라인 끝에서 **EOL 최종검사**를 받습니다.
7월 한 달간 **1,200대 중 48대(4.0%)가 불합격**했습니다.

현장에서는 이런 의견이 나왔습니다.

> "야간조 불량률(5.36%)이 주간조(2.81%)보다 높다. 야간 작업자 교육을 강화하자."

숫자는 맞습니다. 그런데 야간에는 **만든 제품모델과 사용한 설비**도 달랐습니다.
그리고 모든 제품을 두 번 검사할 수는 없으므로, **검사 전에 추가확인할 제품을 고르는 기준**이 필요합니다.

### 핵심 질문
> **야간조 불량률이 높은 이유가 정말 근무조 때문인지, 아니면 야간에 만든 제품모델·설비·작업조건이 달랐기 때문인지 확인하고, EOL 검사 전에 추가확인할 제품을 선별하는 기준을 만든다.**

### 판단 시점
> **제품이 EOL 검사대에 도착하기 직전 (조립은 끝났고 검사결과는 아직 모르는 시점)**

판단 시점이 중요한 이유: **그 시점에 알 수 없는 값**(검사결과 이후에 생기는 값)을
분석에 쓰면, 시험 답안지를 미리 본 것과 같아집니다. (2편에서 자세히 다룹니다.)

---

## 1. 준비 — 라이브러리와 한글 폰트

| 도구 | 역할 |
|---|---|
| `pandas` (pd) | 표 데이터 처리 (엑셀에 해당) |
| `numpy` (np) | 숫자 계산 |
| `matplotlib` (plt) · `seaborn` (sns) | 그래프 |
| `scikit-learn` | 머신러닝 |

아래 셀은 **한 번만** 실행하면 됩니다.

In [ ]:
# [준비 셀] 내용을 자세히 이해할 필요는 없습니다. 실행만 하세요.
# 라이브러리 불러오기 + 한글 폰트 설정
import os, sys, glob, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

# -----------------------------------------------------------------------------
# 한글 폰트 설정
#   ※ 순서 주의: seaborn 스타일을 먼저 적용해야 한다.
#      sns.set_style() 이 글꼴 설정을 sans-serif 로 덮어쓰기 때문이다.
# -----------------------------------------------------------------------------
sns.set_style("whitegrid")                      # ① 스타일 먼저

from matplotlib import font_manager as fm

KOREAN_FONTS = ["NanumGothic", "NanumBarunGothic", "Malgun Gothic", "AppleGothic",
                "Apple SD Gothic Neo", "Noto Sans CJK KR", "Noto Sans KR"]


def find_korean_font():
    """설치된 한글 폰트를 찾고, 없으면 Colab에서 나눔폰트를 설치한다."""
    have = {f.name for f in fm.fontManager.ttflist}
    for n in KOREAN_FONTS:
        if n in have:
            return n
    if IN_COLAB:                                 # 나눔폰트 설치 후 다시 찾기
        print("한글 폰트를 설치하는 중입니다... (20초 정도 걸립니다)")
        os.system("apt-get install -y fonts-nanum > /dev/null 2>&1")
        os.system(f"{sys.executable} -m pip install -q koreanize-matplotlib")
        for path in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            fm.fontManager.addfont(path)
        have = {f.name for f in fm.fontManager.ttflist}
        for n in KOREAN_FONTS:
            if n in have:
                return n
    return None


FONT = find_korean_font()                        # ② 폰트 나중에
if FONT:
    plt.rcParams["font.family"] = FONT
    plt.rcParams["font.sans-serif"] = [FONT] + list(plt.rcParams["font.sans-serif"])
plt.rcParams["axes.unicode_minus"] = False       # 음수 기호(-) 깨짐 방지
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("준비 완료 | pandas", pd.__version__, "| Colab:", IN_COLAB)
print("한글 폰트 :", FONT if FONT else "찾지 못함")

# ③ 자가진단 - 아래 그림에 한글이 보이면 정상입니다
plt.figure(figsize=(5, 0.9))
plt.text(0.5, 0.5, "한글 표시 확인 - 가나다 ABC 123",
         ha="center", va="center", fontsize=15)
plt.axis("off")
plt.show()
print("위 그림의 글자가 □□□ 로 보이면 [런타임 > 런타임 다시 시작] 후 이 셀을 다시 실행하세요.")

---

## 2. 데이터 불러오기

Colab은 구글의 임시 컴퓨터에서 돌아가므로 **데이터 파일을 올려 줘야** 합니다.
아래 셀이 흔한 경로를 먼저 찾아보고, 못 찾으면 **업로드 창**을 띄웁니다.
(드라이브를 쓰려면 맨 위 2줄의 `#` 을 지우세요.)

- 사용할 파일: `K-뉴딜_아카데미_1회차_공통MiniCase_미니프로젝트_데이터셋_수정.xlsx`

In [ ]:
# [데이터 적재 셀] 파일을 찾아서 읽어 오는 코드입니다. 실행만 하세요.
# from google.colab import drive
# drive.mount('/content/drive')

DATA_FILE = "K-뉴딜_아카데미_1회차_공통MiniCase_미니프로젝트_데이터셋_수정.xlsx"
SEARCH_DIRS = [os.environ.get("KNEWDEAL_DATA_DIR", ""), ".", "..", "../..", "/content",
               "/content/drive/MyDrive", "/content/drive/MyDrive/K뉴딜"]

DATA_PATH = None
for d in SEARCH_DIRS:                      # ① 정확한 파일명으로 찾기
    if d and os.path.exists(os.path.join(d, DATA_FILE)):
        DATA_PATH = os.path.join(d, DATA_FILE)
        break
if DATA_PATH is None:                      # ② 키워드로 찾기
    for d in SEARCH_DIRS:
        hits = glob.glob(os.path.join(d, "*MiniCase*")) if d else []
        if hits:
            DATA_PATH = sorted(hits)[0]
            break
if DATA_PATH is None and IN_COLAB:         # ③ 직접 업로드
    from google.colab import files
    DATA_PATH = list(files.upload().keys())[0]

print("데이터 경로:", DATA_PATH)
df_raw = pd.read_excel(DATA_PATH, sheet_name="MiniCase_Data")
print("크기(행, 열):", df_raw.shape)
df_raw.head(3)

---

## 3. 데이터 첫인상

새 데이터를 받으면 **크기 → 자료형 → 요약통계** 순으로 봅니다.
요약통계에서는 **불가능한 값**(음수 수량, 범위를 벗어난 비율)이 있는지 확인합니다.

In [ ]:
df = df_raw.copy()          # 원본은 보존하고 복사본으로 작업

print("행 수:", df.shape[0], " / 열 수:", df.shape[1])
print("\n[자료형과 결측]")
df.info()
print("\n[수치형 요약통계]")
display(df.describe().T.round(2))

---

## 4. 결측값 확인

결측은 **무조건 지우지 않습니다.** 먼저 "왜 없는가"를 봅니다.

| 종류 | 예 | 처리 |
|---|---|---|
| 구조적 결측 | 합격품의 불량유형 | **채우지 않는다** (원래 없는 값) |
| 센서 누락 | 토크·진동 결측 | 대체값으로 채우거나 별도 로그로 복원 |

이 데이터에는 **토크 결측 12건**(별도 센서로그에 복원값 있음),
**진동 결측 18건**(복원자료 없음), **불량유형 결측 1,152건**(합격품이라 원래 없음)이 있습니다.

In [ ]:
miss = pd.DataFrame({"결측건수": df.isna().sum(),
                     "결측비율(%)": (df.isna().mean() * 100).round(2)})
display(miss[miss["결측건수"] > 0])

print("record_id 중복:", df["record_id"].duplicated().sum(), "건  (0이어야 정상)")

---

## 5. 목표변수 — 불량은 얼마나 드문가

찾아내고 싶은 사건(불합격(추가확인 대상))을 **1**, 나머지를 **0** 으로 둡니다.

**가장 중요한 포인트**: 양성이 드물면 **"전부 음성"이라고만 답해도 정확도가 매우 높습니다.**
그래서 이런 데이터에서는 정확도만 보면 안 됩니다.

In [ ]:
df["target"] = (df['inspection_result'] == '불합격').astype(int)       # 1 = 불합격(추가확인 대상)

print(df["inspection_result"].value_counts().to_string())
print(f"\n양성(1) 비율         : {df['target'].mean()*100:.2f}%")
print(f"'전부 음성' 모델의 정확도 : {(1-df['target'].mean())*100:.2f}%  ← 이것을 넘어야 의미가 있다")

cnt = df["target"].value_counts().sort_index()
plt.figure(figsize=(5, 3.5))
plt.bar(["0: 합격", "1: 불합격(추가확인 대상)"], cnt.values, color=["#4e79a7", "#e15759"])
for i, v in enumerate(cnt.values):
    plt.text(i, v, f"{v}건 ({v/len(df)*100:.1f}%)", ha="center", va="bottom")
plt.ylim(0, cnt.values.max() * 1.2)
plt.title("목표변수 분포")
plt.show()

---

## 6. 그룹별 불량률 비교

범주형 변수로 나눠 **그룹별 불량률**을 봅니다. 현장 보고에 바로 쓰이는 분석입니다.

**반드시 표본 수를 함께 보세요.** "10건 중 2건(20%)"과 "400건 중 40건(10%)"은
비율만 보면 앞이 2배 높지만, 앞은 1건만 달라져도 비율이 크게 흔들립니다.
→ 항상 **"n건 중 m건(x%)"** 형식으로 적습니다.

In [ ]:
def group_rate(col):
    """그룹별 전체건수 / 양성건수 / 양성률(%) 표"""
    g = df.groupby(col)["target"]
    t = pd.DataFrame({"전체건수": g.size(), "양성건수": g.sum().astype(int)})
    t["양성률(%)"] = (t["양성건수"] / t["전체건수"] * 100).round(2)
    return t


GROUPS = ["shift", "line_id", "product_model", "equipment_id"]
for c in GROUPS:
    print(f"===== {c} =====")
    display(group_rate(c))

In [ ]:
# 그룹별 불량률 그래프 (회색 점선 = 전체 평균)
overall = df["target"].mean() * 100
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, c in zip(axes.ravel(), GROUPS):
    t = group_rate(c)
    ax.bar(t.index.astype(str), t["양성률(%)"], color="#e15759")
    ax.axhline(overall, color="gray", ls="--")
    for i, (n, m, r) in enumerate(zip(t["전체건수"], t["양성건수"], t["양성률(%)"])):
        ax.text(i, r, f"{r:.1f}%\n({m}/{n})", ha="center", va="bottom", fontsize=8)
    ax.set_title(f"{c} 별 불량률")
    ax.set_ylim(0, t["양성률(%)"].max() * 1.4)
plt.tight_layout()
plt.show()

---

## 7. 교차표 — 두 조건을 겹쳐서 보기

그룹을 **하나씩** 볼 때 보이는 차이가, 사실은 **다른 변수 때문**일 수 있습니다.
이것을 **교란(confounding)** 이라고 합니다.

예: "야간조 불량률이 높다" → 야간에 **어떤 설비·제품을 썼는지** 함께 봐야 합니다.
`shift` 와 `equipment_id` 를 겹쳐서 확인해 봅시다.

In [ ]:
GX, GY = "shift", "equipment_id"

cnt = pd.crosstab(df[GX], df[GY])
pos = pd.crosstab(df[GX], df[GY], values=df["target"], aggfunc="sum").fillna(0)
rate = (pos / cnt * 100).round(1)

plt.figure(figsize=(7, 3.2))
annot = rate.astype(str) + "%\n(" + pos.astype(int).astype(str) + "/" + cnt.astype(str) + ")"
sns.heatmap(rate, annot=annot, fmt="", cmap="Reds", linewidths=0.5)
plt.title(f"{GX} × {GY} 불량률")
plt.show()

display(rate)

### 이 표에서 확인할 것

- **야간 + M03 조합**: 140건 중 15건(10.7%) 불합격. 그 외 조건은 1,060건 중 33건(3.1%).
  → "야간이 문제"라기보다 **"야간에 많이 돌린 M03"** 이 더 설명력이 높습니다.
- **라인과 제품모델은 완전히 겹칩니다** (L1=P1, L2=P2). 따라서 "L2 때문"과 "P2 때문"을
  이 데이터만으로는 **구분할 수 없습니다.**
- 작업그룹별 차이도 **사람의 능력 차이가 아닙니다.** 그룹마다 담당한 설비·제품이 다릅니다.

---

## 8. 공정규격과 실제 분포 — 체결토크 교육용 관리규격 27~33 N·m

제조 데이터에서는 **통계적 이상치**와 **공정규격 이탈**이 다릅니다.

- 통계적 이상치: 분포에서 멀리 떨어진 값
- **공정규격 이탈**: 관리기준(27~33)을 벗어난 값 ← 현장에서 중요한 것은 이쪽

히스토그램에 규격선을 그어 **규격을 벗어난 제품이 모두 불량인지** 확인해 봅시다.

In [ ]:
COL, LO, HI = "torque_nm", 27, 33

plt.figure(figsize=(9, 4))
plt.hist(df.loc[df["target"] == 0, COL].dropna(), bins=35, color="#9bbcd6", label="합격")
plt.hist(df.loc[df["target"] == 1, COL].dropna(), bins=35, color="#e15759", label="불합격")
plt.axvline(LO, color="red", ls="--", lw=2, label=f"규격 {LO}~{HI}")
plt.axvline(HI, color="red", ls="--", lw=2)
plt.xlabel(COL); plt.ylabel("건수"); plt.legend()
plt.title(f"{COL} 분포와 공정규격")
plt.show()

out = ((df[COL] < LO) | (df[COL] > HI))
print("[규격 이탈 여부 × 검사결과]")
display(pd.crosstab(out.map({True: "규격 밖", False: "규격 안"}), df["inspection_result"]))

---

## 9. 수치형 변수 비교 — 불량품은 무엇이 다른가

양성(불합격)과 음성(합격)의 **분포 차이**를 봅니다.
상자가 많이 겹치면 **그 변수 하나로는 구분할 수 없다**는 뜻입니다.
제조 데이터에서는 이런 경우가 대부분이고, 그래서 **여러 변수를 조합하는 머신러닝**이 필요합니다.

In [ ]:
NUM_COLS = ["torque_nm", "temperature_c", "vibration_mm_s", "cycle_time_sec"]

fig, axes = plt.subplots(1, len(NUM_COLS), figsize=(3.2 * len(NUM_COLS), 3.6))
for ax, c in zip(axes.ravel(), NUM_COLS):
    data = [df.loc[df["target"] == 0, c].dropna(), df.loc[df["target"] == 1, c].dropna()]
    bp = ax.boxplot(data, patch_artist=True)
    for patch, color in zip(bp["boxes"], ["#4e79a7", "#e15759"]):
        patch.set_facecolor(color)
    ax.set_xticks([1, 2]); ax.set_xticklabels(["합격", "불합격"])
    ax.set_title(c, fontsize=10)
plt.tight_layout()
plt.show()

# 평균 비교표
cmp = pd.DataFrame({
    "합격 평균": df[df["target"] == 0][NUM_COLS].mean(),
    "불합격 평균": df[df["target"] == 1][NUM_COLS].mean(),
})
cmp["차이"] = cmp["불합격 평균"] - cmp["합격 평균"]
display(cmp.round(3))

---

## 10. 산점도 — 점 하나가 제품 하나

두 변수를 축에 놓고 색으로 합격/불합격을 구분하면,
**불량이 특정 영역에 모여 있는지** 눈으로 확인할 수 있습니다.
모여 있다면 그것이 **판정규칙의 후보**가 됩니다.

In [ ]:
X, Y = "torque_nm", "temperature_c"

plt.figure(figsize=(7, 5))
neg, pos_ = df[df["target"] == 0], df[df["target"] == 1]
plt.scatter(neg[X], neg[Y], s=14, alpha=0.3, c="#4e79a7", label="합격")
plt.scatter(pos_[X], pos_[Y], s=45, c="#e15759", edgecolors="black",
            linewidths=0.4, label="불합격")
plt.xlabel(X); plt.ylabel(Y); plt.legend()
plt.title(f"{X} vs {Y}")
plt.show()

print("불합격(빨강)이 한쪽에 모여 있나요? 흩어져 있다면 이 두 변수만으로는 구분이 어렵습니다.")

---

## 11. 정리

- 불량률 4.0% → **모두 합격이라고 찍어도 정확도 96%**. 정확도만 보면 안 된다.
- 설비 **M03의 불량률이 7.33%** 로 가장 높고, 특히 **야간 M03이 10.7%** 다.
- 토크 **규격 밖 100건 중 80건은 합격**이다. 규격 이탈이 곧 불량은 아니다.
  반대로 불합격 48건 중 28건은 규격 안에 있다 → 토크 하나로는 부족하다.
- 압력·사이클타임·에너지는 합격/불합격 차이가 거의 없다. **변수가 있다고 쓸모 있는 건 아니다.**
- 야간조는 **원인이 아니라 추가확인이 필요한 조건**이다.

### 2편으로 가져갈 것
1. 결측 처리 방침 (어떻게 채울지)
2. **검사 전에 알 수 있는 변수 목록** (모델 입력 후보)
3. "어떤 조건이 위험한가"에 대한 가설 2~3개

> 1편의 결론은 **"무엇이 원인이다"가 아니라 "어디를 먼저 확인해야 한다"** 입니다.

---

## 연습문제 / 생각해 볼 질문

아래 문항은 정답을 맞히는 시험이 아니라, **발표자료에 쓸 문장을 만드는 연습**입니다.
코드를 조금씩 바꿔 보고, 결과를 한 문장으로 정리해 보세요.

1. `group_rate("operator_group")` 를 실행해 작업그룹별 불량률을 보고, 가장 높은 그룹의 **표본 수**를 확인하세요. 이 차이를 사람의 능력 차이로 해석하면 안 되는 이유는?
2. 교차표에서 **야간 M03을 제외**하면 야간조와 주간조의 불량률 차이가 얼마나 줄어드는지 계산해 보세요.
3. 토크 규격 밖 100건 중 80건이 합격입니다. 토크 기준만으로 추가검사를 하면 어떤 문제가 생길까요? 한 문장으로 쓰세요.